## Proyecto GT - Fase 2 - Analisis



In [ ]:
#Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from kedro.framework.session import KedroSession

session = KedroSession.create()
context = session.load_context()
datos = context.catalog.load("datos_crudos")


In [ ]:
# Carga del dataset (asegúrate de tener el CSV en la misma carpeta que este notebook)
csv_path = "gaming.csv"
assert os.path.exists(csv_path), "No se encontró el archivo Gaming-Trends-2024.csv en el directorio actual."

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Columnas:", list(df.columns))
df.head()


In [ ]:
# Tipos y valores faltantes
display(df.dtypes)
na_counts = df.isna().sum().sort_values(ascending=False)
display(na_counts.to_frame(name="missing"))

# Duplicados
print("Duplicados:", df.duplicated().sum())

In [ ]:
def find_col(df, key_candidates):
    # Try to find a column in df whose lowercased name contains ALL tokens in any candidate string.
    # Returns the best match or raises ValueError if not found.
    cols = list(df.columns)
    lower = [c.lower() for c in cols]
    scores = []
    for cand in key_candidates:
        tokens = [t.strip() for t in cand.lower().replace("(", " ").replace(")", " ").replace("$"," ").split() if t.strip()]
        for i, lc in enumerate(lower):
            if all(tok in lc for tok in tokens):
                scores.append((i, cols[i], len(tokens)))
                print(f"find_col: candidate '{cand}' matches column '{cols[i]}'")
    if not scores:
        raise ValueError(f"No se encontró ninguna columna para {key_candidates}")
    # prefer the match with more tokens (more specific)
    scores.sort(key=lambda x: (-x[2], x[0]))
    return scores[0][1]

In [ ]:
# Normalizamos/parseamos la fecha
try:
    date_col = find_col(df, ["date", "fecha"])
except Exception as e:
    raise

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
assert df[date_col].notna().any(), "La columna de fecha no pudo parsearse. Revisa el formato."

df.describe


### Estadísticos descriptivos (tendencia central y dispersión)
Incluye: media, mediana, moda, varianza, desviación estándar, rango, coeficiente de variación (CV), IQR, mínimos/máximos, suma y conteos.


In [ ]:
def tabla_descriptiva(d): # función para generar tabla descriptiva
    num = d.select_dtypes(include=[np.number]).copy() # seleccionar solo columnas numéricas
    summary = pd.DataFrame(index=num.columns) # crear DataFrame vacío con índices como nombres de columnas numéricas
    # Cálculo de estadísticas
    summary["count"] = num.count()
    summary["sum"] = num.sum()
    summary["mean"] = num.mean()
    summary["median"] = num.median()
    summary["mode"] = num.mode().iloc[0] if not num.mode().empty else np.nan
    summary["var"] = num.var(ddof=1)
    summary["std"] = num.std(ddof=1)
    summary["min"] = num.min()
    summary["q1"] = num.quantile(0.25)
    summary["q3"] = num.quantile(0.75)
    summary["iqr"] = summary["q3"] - summary["q1"]
    summary["max"] = num.max()
    summary["range"] = summary["max"] - summary["min"]
    summary["cv"] = summary["std"] / summary["mean"]
    return summary

desc = tabla_descriptiva(df)
display(desc.round(3))
